# Phase 04L: Teacher B → S2 Sequence-KD Seed-2026

Confirmation run với phương pháp đã khóa từ seed 42:

- chỉ thay development seed thành `2026`;
- giữ nguyên pseudo corpus Teacher B seed 42;
- giữ nguyên loss, optimizer, scheduler, batch, augmentation và Validation;
- so sánh với S2 CE-only baseline seed 2026;
- Test luôn `SEALED`;
- hỗ trợ atomic resume và session budget 10,5 giờ.

## Inputs

1. Phase 01 formal output.
2. Phase 02A shared Student bundle.
3. Phase 02C S2 baseline seed 2026.
4. Phase 04I pseudo corpus.
5. Phase 04J readiness output.
6. Phase 04K seed-42 complete output.
7. MS COCO 2014 images.
8. Offline `pycocoevalcap==1.2` wheel.

Khi resume, attach thêm output Phase 04L seed 2026 mới nhất.


## Cell 1 — Authorization, matched seed 2026 và resume source

Xác minh Phase 04K, confirmation authorization, production contract,
pseudo corpus và S2 baseline đúng seed 2026.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 2026
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_b_s2_sequence_kd_seed2026"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RESULT_DIR = OUTPUT_ROOT / "results"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, LOG_DIR, RESULT_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04L bắt buộc GPU. Chọn GPU T4 hoặc mạnh hơn rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE4J_COMPLETE = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "TRAINING_READINESS_COMPLETE.json"
)
FULL_RUN_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "SEED42_FULL_RUN_AUTHORIZED.json"
)
CONFIRMATION_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "CONFIRMATION_SEEDS_AUTHORIZED.json"
)
DEVELOPMENT_RUN_COMPLETE = exactly_one(
    "PHASE4_TEACHER_B_S2_SEQUENCE_KD_"
    "SEED42_FULL_RUN_COMPLETE.json"
)
PHASE4I_COMPLETE = exactly_one(
    "PHASE4_TEACHER_B_S2_PSEUDO_CORPUS_COMPLETE.json"
)
PHASE2_READY = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
S2_BASELINE_COMPLETE = exactly_one(
    "S2_SEED2026_FULL_RUN_COMPLETE.json"
)

phase4j_complete = read_json(PHASE4J_COMPLETE)
full_run_authorized = read_json(FULL_RUN_AUTHORIZED)
confirmation_authorized = read_json(CONFIRMATION_AUTHORIZED)
development_run = read_json(DEVELOPMENT_RUN_COMPLETE)
phase4i_complete = read_json(PHASE4I_COMPLETE)
phase2_ready = read_json(PHASE2_READY)
phase1_ready = read_json(PHASE1_READY)
s2_baseline = read_json(S2_BASELINE_COMPLETE)

assert phase4j_complete["status"] == "complete"
assert phase4j_complete["production_data_pipeline"] == "passed"
assert phase4j_complete["model_optimizer_engine"] == "passed"
assert phase4j_complete["validation_beam_engine"] == "passed"
assert phase4j_complete["discarded_pilot"] == "passed"
assert phase4j_complete["production_contract"] == "locked"
assert phase4j_complete["phase04k_full_run_authorized"] is True
assert full_run_authorized["status"] == "authorized"
assert full_run_authorized["experiment_id"] == "P4B_B_S2_SEQKD"
assert int(full_run_authorized["seed"]) == 42
assert int(full_run_authorized["pseudo_corpus"]["rows"]) == 113287
assert confirmation_authorized["status"] == "authorized"
assert confirmation_authorized["experiment_id"] == "P4B_B_S2_SEQKD"
assert int(confirmation_authorized["development_seed_complete"]) == 42
assert 2026 in [
    int(value)
    for value in confirmation_authorized["confirmation_seeds"]
]
assert confirmation_authorized["method_changes_allowed"] is False
assert development_run["status"] == "complete"
assert int(development_run["seed"]) == 42
assert development_run["confirmation_seeds_authorized"] is True
assert phase4i_complete["status"] == "complete"
assert s2_baseline["status"] == "complete"
assert s2_baseline["student_id"] == "S2"
assert int(s2_baseline["seed"]) == 2026
assert phase2_ready["status"] == "ready"
assert phase1_ready["status"] == "ready"
for marker in [
    phase4j_complete, full_run_authorized,
    confirmation_authorized, development_run,
    phase4i_complete, s2_baseline,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False

PHASE4J_ROOT = PHASE4J_COMPLETE.parent
production_ref = full_run_authorized["production_contract"]
PRODUCTION_CONTRACT_PATH = resolve_inside(
    PHASE4J_ROOT, production_ref["relative_path"]
)
assert sha256_file(PRODUCTION_CONTRACT_PATH) == (
    production_ref["sha256"]
)
PRODUCTION_CONTRACT = read_json(PRODUCTION_CONTRACT_PATH)
assert PRODUCTION_CONTRACT["status"] == "locked"
assert PRODUCTION_CONTRACT["kd"]["channel"] == (
    "sequence_pseudo_caption"
)
assert PRODUCTION_CONTRACT["kd"]["temperature"] is None
assert float(PRODUCTION_CONTRACT["kd"]["alpha"]) == 0.5

phase4j_manifest = resolve_inside(
    PHASE4J_ROOT,
    phase4j_complete["manifest"]["csv"]["relative_path"],
)
phase4j_sums = resolve_inside(
    PHASE4J_ROOT,
    phase4j_complete["manifest"]["sha256sums"]["relative_path"],
)
rows = list(csv.DictReader(
    phase4j_manifest.open("r", encoding="utf-8", newline="")
))
assert len(rows) == 11
expected_lines = []
for row in rows:
    artifact = resolve_inside(PHASE4J_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
    expected_lines.append(
        f'{row["sha256"]}  {row["relative_path"]}'
    )
actual_lines = [
    line.strip()
    for line in phase4j_sums.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]
assert actual_lines == expected_lines

PHASE4I_ROOT = PHASE4I_COMPLETE.parent
PSEUDO_CORPUS = resolve_inside(
    PHASE4I_ROOT,
    phase4i_complete["pseudo_corpus"]["relative_path"],
)
assert sha256_file(PSEUDO_CORPUS) == (
    phase4i_complete["pseudo_corpus"]["sha256"]
)
assert sha256_file(PSEUDO_CORPUS) == (
    full_run_authorized["pseudo_corpus"]["source_sha256"]
)
assert sha256_file(PSEUDO_CORPUS) == (
    PRODUCTION_CONTRACT["teacher"]["pseudo_corpus_sha256"]
)


PHASE4K_ROOT = DEVELOPMENT_RUN_COMPLETE.parent
development_manifest = resolve_inside(
    PHASE4K_ROOT,
    development_run["manifest"]["csv"]["relative_path"],
)
development_sums = resolve_inside(
    PHASE4K_ROOT,
    development_run["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(development_manifest) == (
    development_run["manifest"]["csv"]["sha256"]
)
assert sha256_file(development_sums) == (
    development_run["manifest"]["sha256sums"]["sha256"]
)
development_rows = list(csv.DictReader(
    development_manifest.open(
        "r", encoding="utf-8", newline=""
    )
))
assert len(development_rows) == 53
development_expected = []
for row in development_rows:
    artifact = resolve_inside(PHASE4K_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
    development_expected.append(
        f'{row["sha256"]}  {row["relative_path"]}'
    )
development_actual = [
    line.strip()
    for line in development_sums.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]
assert development_actual == development_expected

BASELINE_CIDER = float(
    s2_baseline["best_validation_metrics"]["CIDEr-D"]
)
BASELINE_BLEU4 = float(
    s2_baseline["best_validation_metrics"]["BLEU-4"]
)
BASELINE_CE = float(
    s2_baseline["best_validation_cross_entropy"]
)

resume_candidates = sorted(
    INPUT_ROOT.rglob(
        "teacher_b_s2_sequence_kd_seed2026_last.pt"
    )
)
assert len(resume_candidates) <= 1, resume_candidates
INPUT_RESUME_CHECKPOINT = (
    resume_candidates[0].resolve()
    if resume_candidates else None
)

write_json(
    AUDIT_DIR / "phase4l_seed2026_cell01_authorization.json",
    {
        "audit_version": "phase4l_seed2026_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase4j_manifest_rows": len(rows),
        "phase4k_development_manifest_rows": len(development_rows),
        "development_seed42_run": "verified",
        "confirmation_seed": 2026,
        "confirmation_authorization": "verified",
        "method_changes_allowed": False,
        "production_contract_sha256": sha256_file(
            PRODUCTION_CONTRACT_PATH
        ),
        "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
        "s2_seed42_baseline": {
            "cider_d": BASELINE_CIDER,
            "bleu_4": BASELINE_BLEU4,
            "validation_ce": BASELINE_CE,
        },
        "resume_checkpoint_found": (
            INPUT_RESUME_CHECKPOINT is not None
        ),
        "dataset_read": False,
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04J manifest        : 11 / 11 VERIFIED")
print("Full run authorization    : VERIFIED")
print("Seed42 development run    : VERIFIED")
print("Confirmation authorization: VERIFIED")
print("Method changes allowed    : False")
print("Production contract       : LOCKED")
print("Pseudo corpus             : VERIFIED")
print("S2 seed2026 baseline CIDEr-D:", BASELINE_CIDER)
print("Resume checkpoint found   :", INPUT_RESUME_CHECKPOINT is not None)
print("GPU                       :", torch.cuda.get_device_name(0))
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 cells passed")
print("Next                      : rebuild production data pipeline")


PHASE 04L SEED 2026 CELL 1 STATUS
Status                    : PASSED
Phase 04J manifest        : 11 / 11 VERIFIED
Full run authorization    : VERIFIED
Seed42 development run    : VERIFIED
Confirmation authorization: VERIFIED
Method changes allowed    : False
Production contract       : LOCKED
Pseudo corpus             : VERIFIED
S2 seed2026 baseline CIDEr-D: 0.9999397940007319
Resume checkpoint found   : False
GPU                       : Tesla T4
Checkpoint payload read   : False
Test Set status           : SEALED
Progress                  : 1 / 6 cells passed
Next                      : rebuild production data pipeline


## Cell 2 — Rebuild production datasets and sampler


In [2]:
CELL_STARTED_AT = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
    filters=[("experiment_split", "in", ["train", "validation"])],
)
TRAIN_PATHS = {}
VALIDATION_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    image_path = leaf / Path(row["image_path"]).name
    target = (
        TRAIN_PATHS
        if row["experiment_split"] == "train"
        else VALIDATION_PATHS
    )
    target[image_id] = image_path
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_PATHS) == 5000

TRAIN_CAPTIONS = defaultdict(list)
for arrow_batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in arrow_batch.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append(
            (
                int(row["annotation_id"]),
                tuple(int(value) for value in row["token_ids"]),
            )
        )
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()
assert len(TRAIN_CAPTIONS) == 113287

PSEUDO_BY_IMAGE = {}
pseudo_file = pq.ParquetFile(PSEUDO_CORPUS)
assert pseudo_file.metadata.num_rows == 113287
assert set(pseudo_file.schema_arrow.names) == {
    "image_id", "teacher_caption", "student_token_ids",
    "student_lexical_tokens", "unknown_tokens", "truncated",
    "teacher_seed", "split",
}
for arrow_batch in pseudo_file.iter_batches(
    batch_size=16384,
    columns=[
        "image_id", "student_token_ids", "teacher_seed", "split"
    ],
):
    for row in arrow_batch.to_pylist():
        token_ids = tuple(int(value) for value in row["student_token_ids"])
        assert token_ids[0] == 1 and token_ids[-1] == 2
        assert len(token_ids) <= 32
        assert min(token_ids) >= 0 and max(token_ids) < 10080
        assert int(row["teacher_seed"]) == 42
        assert row["split"] == "Train+Restval"
        image_id = int(row["image_id"])
        assert image_id not in PSEUDO_BY_IMAGE
        PSEUDO_BY_IMAGE[image_id] = token_ids
assert set(PSEUDO_BY_IMAGE) == set(TRAIN_PATHS)

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for arrow_batch in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in arrow_batch.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append(
            (
                int(row["annotation_id"]),
                image_id,
                tuple(int(value) for value in row["token_ids"]),
            )
        )
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))


def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|phase4l_seed2026"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8], "big"
    ) % (2 ** 63 - 1)


def train_transform(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = (
            transforms.RandomResizedCrop.get_params(
                image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
            )
        )
        do_flip = bool(torch.rand(()) < 0.5)
    image = TF.resized_crop(
        image, top, left, height, width, [224, 224],
        InterpolationMode.BICUBIC, antialias=True,
    )
    if do_flip:
        image = TF.hflip(image)
    return TF.to_tensor(image)


def validation_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    return TF.to_tensor(TF.center_crop(image, [224, 224]))


class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, human_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as source:
            image = train_transform(
                source.convert("RGB"), self.epoch, image_id
            )
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "human_ids": torch.tensor(human_ids, dtype=torch.long),
            "pseudo_ids": torch.tensor(
                PSEUDO_BY_IMAGE[image_id], dtype=torch.long
            ),
        }


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as source:
            image = validation_transform(source.convert("RGB"))
        return {"image_id": image_id, "student_image": image}


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(
            self.length, generator=generator
        ).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor


def pad_sequences(sequences):
    maximum = max(item.numel() for item in sequences)
    result = torch.zeros(len(sequences), maximum, dtype=torch.long)
    for index, item in enumerate(sequences):
        result[index, :item.numel()] = item
    return result


def train_collate(rows):
    human = pad_sequences([row["human_ids"] for row in rows])
    pseudo = pad_sequences([row["pseudo_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "human_input_ids": human[:, :-1],
        "human_labels": human[:, 1:],
        "human_mask": human[:, :-1].ne(0),
        "pseudo_input_ids": pseudo[:, :-1],
        "pseudo_labels": pseudo[:, 1:],
        "pseudo_mask": pseudo[:, :-1].ne(0),
    }


def validation_caption_collate(rows):
    full = pad_sequences([row["token_ids"] for row in rows])
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
        "decoder_attention_mask": full[:, :-1].ne(0),
    }


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }


TRAIN_DATASET = ProductionTrainDataset()
VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()


def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    return DataLoader(
        TRAIN_DATASET,
        batch_size=32,
        sampler=EpochCursorSampler(
            len(TRAIN_DATASET), epoch=epoch, cursor=cursor
        ),
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=train_collate,
    )


VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=validation_caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

batch_a = next(iter(build_train_loader(0)))
batch_b = next(iter(build_train_loader(0)))
for key in [
    "image_ids", "annotation_ids", "student_images",
    "human_input_ids", "human_labels",
    "pseudo_input_ids", "pseudo_labels",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert len(set(batch_a["image_ids"].tolist())) == 32
assert 0.0 <= float(batch_a["student_images"].min())
assert float(batch_a["student_images"].max()) <= 1.0

write_json(
    AUDIT_DIR / "phase4l_seed2026_cell02_data_pipeline.json",
    {
        "audit_version": "phase4l_seed2026_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_images_per_epoch": len(TRAIN_DATASET),
        "human_captions_per_image_per_epoch": 1,
        "teacher_sequences_per_image_per_epoch": 1,
        "pseudo_membership_exact": True,
        "validation_captions": len(VALIDATION_CAPTION_DATASET),
        "validation_images": len(VALIDATION_IMAGE_DATASET),
        "physical_batch_size": 32,
        "normalization": "mean 0 std 1; tensor range [0,1]",
        "stateless_augmentation": True,
        "two_loader_exact": True,
        "resume_cursor_supported": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train images / epoch      :", len(TRAIN_DATASET))
print("Human captions / image    : 1, cyclic")
print("Teacher sequence / image  : 1, fixed")
print("Pseudo membership exact   : True")
print("Validation captions       :", len(VALIDATION_CAPTION_DATASET))
print("Validation images         :", len(VALIDATION_IMAGE_DATASET))
print("Physical/effective batch  : 32 / 64")
print("S2 input range            : [0,1]")
print("Two-loader exact          : True")
print("Resume cursor supported   : True")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 cells passed")
print("Next                      : S2 model and optimizer")


PHASE 04L SEED 2026 CELL 2 STATUS
Status                    : PASSED
Train images / epoch      : 113287
Human captions / image    : 1, cyclic
Teacher sequence / image  : 1, fixed
Pseudo membership exact   : True
Validation captions       : 25010
Validation images         : 5000
Physical/effective batch  : 32 / 64
S2 input range            : [0,1]
Two-loader exact          : True
Resume cursor supported   : True
Test Set status           : SEALED
Progress                  : 2 / 6 cells passed
Next                      : S2 model and optimizer


## Cell 3 — Rebuild S2, optimizer và Sequence-KD engine


In [3]:
CELL_STARTED_AT = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion


PHASE2_ROOT = PHASE2_READY.parent
S2_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s2_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080


class S2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = timm.create_model(
            "mobilevit_s.cvnets_in1k", pretrained=False
        )
        if hasattr(backbone, "reset_classifier"):
            backbone.reset_classifier(num_classes=0, global_pool="")
        if hasattr(backbone, "head"):
            backbone.head = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Linear(640, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone.forward_features(images)
        if isinstance(feature, (tuple, list)):
            feature = [x for x in feature if torch.is_tensor(x)][-1]
        if isinstance(feature, dict):
            feature = [
                x for x in feature.values() if torch.is_tensor(x)
            ][-1]
        assert feature.shape[1:] == (640, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )


STUDENT_ENCODER = S2Encoder()
STUDENT_DECODER = SharedDecoder(decoder_config)
with torch.serialization.safe_globals([TorchVersion]):
    encoder_payload = torch.load(
        S2_INITIAL_CHECKPOINT, map_location="cpu", weights_only=True
    )
    decoder_payload = torch.load(
        DECODER_INITIAL_CHECKPOINT,
        map_location="cpu", weights_only=True,
    )
STUDENT_ENCODER.load_state_dict(
    encoder_payload["state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    decoder_payload["state_dict"], strict=True
)
del encoder_payload, decoder_payload
assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 5114784
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5759328

STUDENT_ENCODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
STUDENT_ENCODER.to(device)
STUDENT_DECODER.to(device)


def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 32


def student_logits(batch):
    memory = STUDENT_ENCODER(batch["student_images"])
    return STUDENT_DECODER(
        batch["decoder_input_ids"],
        memory,
        batch["decoder_attention_mask"],
    )


ALPHA = 0.5
TEMPERATURE = None
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 2
STEPS_PER_EPOCH = math.ceil(113287 / 64)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def sequence_sum(logits, labels):
    return F.cross_entropy(
        logits.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=0,
        reduction="sum",
    )


def optimizer_step(
    microbatches, optimizer, scheduler, scaler, retry_limit=10
):
    human_total = sum(
        int(batch["human_labels"].ne(0).sum())
        for batch in microbatches
    )
    pseudo_total = sum(
        int(batch["pseudo_labels"].ne(0).sum())
        for batch in microbatches
    )
    assert human_total > 0 and pseudo_total > 0
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        human_sum_total = 0.0
        pseudo_sum_total = 0.0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                memory = STUDENT_ENCODER(current["student_images"])
                human_logits = STUDENT_DECODER(
                    current["human_input_ids"],
                    memory,
                    current["human_mask"],
                )
                pseudo_logits = STUDENT_DECODER(
                    current["pseudo_input_ids"],
                    memory,
                    current["pseudo_mask"],
                )
                human_sum = sequence_sum(
                    human_logits, current["human_labels"]
                )
                pseudo_sum = sequence_sum(
                    pseudo_logits, current["pseudo_labels"]
                )
                normalized = (
                    (1 - ALPHA) * human_sum / human_total
                    + ALPHA * pseudo_sum / pseudo_total
                )
            scaler.scale(
                normalized / BACKWARD_DIVISOR
            ).backward()
            human_sum_total += float(human_sum)
            pseudo_sum_total += float(pseudo_sum)
        scaler.unscale_(optimizer)
        for parameter in (
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters())
        ):
            if parameter.grad is not None:
                parameter.grad.mul_(BACKWARD_DIVISOR)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            1.0,
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        assert torch.isfinite(gradient_norm)
        scheduler.step()
        human_ce = human_sum_total / human_total
        pseudo_ce = pseudo_sum_total / pseudo_total
        return {
            "combined_loss": (
                (1 - ALPHA) * human_ce + ALPHA * pseudo_ce
            ),
            "human_ce": human_ce,
            "teacher_sequence_ce": pseudo_ce,
            "human_tokens": human_total,
            "teacher_tokens": pseudo_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771

write_json(
    CONFIG_DIR / "phase4l_seed2026_production_optimizer_contract.json",
    {
        "contract_version": "phase4l_seed2026_optimizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "optimizer": "AdamW",
        "learning_rates": {
            "backbone": 1e-4,
            "adapter": 3e-4,
            "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "maximum_epochs": MAX_EPOCHS,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16 plus GradScaler",
        "gradscaler_initial_scale": 1024.0,
        "overflow_retry_limit": 10,
        "backward_divisor": BACKWARD_DIVISOR,
        "gradient_clip": 1.0,
        "alpha": ALPHA,
        "temperature": None,
        "loss": "0.5 CE_human + 0.5 CE_teacher_sequence",
        "normalization": "each channel by its own valid-token count",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4l_seed2026_cell03_model_optimizer.json",
    {
        "audit_version": "phase4l_seed2026_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_loads": 2,
        "teacher_model_loaded": False,
        "pseudo_corpus_fixed": True,
        "student_parameters": 10874112,
        "optimizer_parameter_groups": 3,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "scheduler_advanced": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint loads   : 2 / 2 PASSED")
print("Teacher model loaded      : False")
print("Pseudo corpus fixed       : True")
print("S2 parameters             : 10874112")
print("Optimizer groups          : 3 / 3")
print("Physical/effective batch  : 32 / 64")
print("Steps / epoch             :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("FP16 / GradScaler         : ENABLED / 1024")
print("Overflow retry limit      : 10")
print("Combined loss             : 0.5 human + 0.5 teacher sequence")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 6 cells passed")
print("Next                      : Validation and Beam-3 engine")


PHASE 04L SEED 2026 CELL 3 STATUS
Status                    : PASSED
Strict checkpoint loads   : 2 / 2 PASSED
Teacher model loaded      : False
Pseudo corpus fixed       : True
S2 parameters             : 10874112
Optimizer groups          : 3 / 3
Physical/effective batch  : 32 / 64
Steps / epoch             : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
FP16 / GradScaler         : ENABLED / 1024
Overflow retry limit      : 10
Combined loss             : 0.5 human + 0.5 teacher sequence
Optimizer step performed  : False
Test Set status           : SEALED
Progress                  : 3 / 6 cells passed
Next                      : Validation and Beam-3 engine


## Cell 4 — Full Validation and batched Beam-3 engine


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "phase4l_seed2026_cell04_validation_engine.json",
    {
        "audit_version": "phase4l_seed2026_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : atomic checkpoint and production pilot")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 21360.82 tokens per second.
PTBTokenizer tokenized 385 tokens at 5868.44 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
PHASE 04L SEED 2026 CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 11.258756868131869
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : atomic checkpoint and production pilot


## Cell 5 — Full training seed 2026 hoặc exact resume

Giới hạn session 10,5 giờ; tự tạo trạng thái RESUME READY nếu chưa xong.


In [5]:
CELL_STARTED_AT = time.perf_counter()
from torch.torch_version import TorchVersion


LAST_CHECKPOINT = (
    CHECKPOINT_DIR / "teacher_b_s2_sequence_kd_seed2026_last.pt"
)
BEST_CHECKPOINT = (
    CHECKPOINT_DIR / "teacher_b_s2_sequence_kd_seed2026_best.pt"
)
STEP_LOG = LOG_DIR / "teacher_b_s2_sequence_kd_seed2026_steps.jsonl"
EPOCH_HISTORY_CSV = (
    LOG_DIR / "teacher_b_s2_sequence_kd_seed2026_epoch_history.csv"
)
RUN_STATE_PATH = OUTPUT_ROOT / "PHASE4L_SEED2026_RUN_STATE.json"

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3
VOCABULARY_SIZE = 10080
BEAM_SIZE = 3
LENGTH_PENALTY = 0.7
MAXIMUM_DECODER_STEPS = 31
MINIMUM_LEXICAL_TOKENS = 2
FORBIDDEN_GENERATION_TOKEN_IDS = (PAD_ID, BOS_ID, UNK_ID)


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path, optimizer, scheduler, scaler, progress,
    history, best_result, no_improve_epochs,
):
    atomic_save(
        path,
        {
            "schema_version": "phase4_teacher_b_s2_sequence_kd_checkpoint_v1",
            "experiment_id": "P4B_B_S2_SEQKD",
            "seed": 2026,
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "history": history,
            "best_result": best_result,
            "no_improve_epochs": no_improve_epochs,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "pseudo_corpus_sha256": sha256_file(PSEUDO_CORPUS),
            "production_contract_sha256": sha256_file(
                PRODUCTION_CONTRACT_PATH
            ),
            "temperature": TEMPERATURE,
            "alpha": ALPHA,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


def load_checkpoint(path, optimizer, scheduler, scaler):
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=True
        )
    assert checkpoint["schema_version"] == (
        "phase4_teacher_b_s2_sequence_kd_checkpoint_v1"
    )
    assert checkpoint["experiment_id"] == "P4B_B_S2_SEQKD"
    assert int(checkpoint["seed"]) == 2026
    assert checkpoint["test_set_status"] == "sealed"
    assert checkpoint["test_set_used"] is False
    assert checkpoint["pseudo_corpus_sha256"] == (
        sha256_file(PSEUDO_CORPUS)
    )
    assert checkpoint["production_contract_sha256"] == (
        sha256_file(PRODUCTION_CONTRACT_PATH)
    )
    STUDENT_ENCODER.load_state_dict(
        checkpoint["student_encoder_state"], strict=True
    )
    STUDENT_DECODER.load_state_dict(
        checkpoint["student_decoder_state"], strict=True
    )
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])
    torch.set_rng_state(checkpoint["cpu_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    return (
        checkpoint["progress"],
        checkpoint["history"],
        checkpoint["best_result"],
        int(checkpoint["no_improve_epochs"]),
    )


@torch.inference_mode()
def batched_beam_search(visual_memory):
    batch_size, memory_length, memory_dimension = visual_memory.shape
    assert memory_length == 49 and memory_dimension == 256
    beam_tokens = torch.full(
        (batch_size, BEAM_SIZE, 1),
        BOS_ID, dtype=torch.long, device=device,
    )
    beam_scores = torch.full(
        (batch_size, BEAM_SIZE),
        float("-inf"), dtype=torch.float32, device=device,
    )
    beam_scores[:, 0] = 0.0
    beam_finished = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.bool, device=device
    )
    beam_lengths = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.long, device=device
    )
    expanded_memory = (
        visual_memory.unsqueeze(1)
        .expand(-1, BEAM_SIZE, -1, -1)
        .reshape(batch_size * BEAM_SIZE, memory_length, memory_dimension)
        .contiguous()
    )
    for generation_step in range(MAXIMUM_DECODER_STEPS):
        flat_tokens = beam_tokens.reshape(
            batch_size * BEAM_SIZE, -1
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = STUDENT_DECODER(
                flat_tokens,
                expanded_memory,
                flat_tokens.ne(PAD_ID),
            )
        log_probs = F.log_softmax(
            logits[:, -1, :].float(), dim=-1
        ).reshape(batch_size, BEAM_SIZE, VOCABULARY_SIZE)
        log_probs[:, :, list(FORBIDDEN_GENERATION_TOKEN_IDS)] = (
            float("-inf")
        )
        if generation_step < MINIMUM_LEXICAL_TOKENS:
            log_probs[:, :, EOS_ID] = float("-inf")
        if beam_finished.any():
            log_probs = log_probs.masked_fill(
                beam_finished.unsqueeze(-1), float("-inf")
            )
            finished_indices = beam_finished.nonzero(as_tuple=False)
            log_probs[
                finished_indices[:, 0],
                finished_indices[:, 1],
                EOS_ID,
            ] = 0.0
        candidate_scores = beam_scores.unsqueeze(-1) + log_probs
        next_scores, flat_indices = torch.topk(
            candidate_scores.reshape(
                batch_size, BEAM_SIZE * VOCABULARY_SIZE
            ),
            k=BEAM_SIZE,
            dim=-1,
        )
        parent = flat_indices // VOCABULARY_SIZE
        next_tokens = flat_indices % VOCABULARY_SIZE
        parent_tokens = torch.gather(
            beam_tokens,
            1,
            parent.unsqueeze(-1).expand(
                -1, -1, beam_tokens.shape[-1]
            ),
        )
        parent_finished = torch.gather(
            beam_finished, 1, parent
        )
        parent_lengths = torch.gather(
            beam_lengths, 1, parent
        )
        beam_tokens = torch.cat(
            [parent_tokens, next_tokens.unsqueeze(-1)], dim=-1
        )
        beam_scores = next_scores
        beam_lengths = parent_lengths + (~parent_finished).long()
        beam_finished = parent_finished | (next_tokens == EOS_ID)
        if bool(beam_finished.all()):
            break
    normalized = beam_scores / (
        beam_lengths.clamp_min(1).float() ** LENGTH_PENALTY
    )
    best = normalized.argmax(dim=1)
    batch_indices = torch.arange(batch_size, device=device)
    return {
        "token_ids": beam_tokens[batch_indices, best].cpu(),
        "raw_scores": beam_scores[batch_indices, best].cpu(),
        "normalized_scores": normalized[batch_indices, best].cpu(),
    }


@torch.inference_mode()
def generate_validation_captions():
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, cpu_batch in enumerate(
        VALIDATION_IMAGE_LOADER
    ):
        image_ids = [
            int(value) for value in cpu_batch["image_ids"].tolist()
        ]
        images = cpu_batch["student_images"].to(
            device, non_blocking=True
        )
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = STUDENT_ENCODER(images)
        beam = batched_beam_search(memory)
        for row_index, image_id in enumerate(image_ids):
            token_ids = [
                int(value)
                for value in beam["token_ids"][row_index].tolist()
            ]
            caption = decode_tokens(token_ids)
            assert caption
            assert image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "caption": caption,
                "token_ids": token_ids,
                "raw_score": float(
                    beam["raw_scores"][row_index]
                ),
                "normalized_score": float(
                    beam["normalized_scores"][row_index]
                ),
            }
        if (batch_index + 1) % 25 == 0:
            print(
                f"Generated {len(hypotheses):,}/5,000 "
                "Validation captions"
            )
    assert len(hypotheses) == 5000
    assert set(hypotheses) == set(VALIDATION_REFERENCES)
    return hypotheses, records, time.perf_counter() - started


def evaluate_validation(epoch):
    ce, valid_tokens = teacher_forced_ce(
        VALIDATION_CAPTION_LOADER
    )
    hypotheses, records, generation_seconds = (
        generate_validation_captions()
    )
    metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, hypotheses
    )
    result = {
        "epoch": int(epoch),
        "teacher_forced_ce": float(ce),
        "valid_tokens": int(valid_tokens),
        "metrics": metrics,
        "prediction_count": len(hypotheses),
        "generation_seconds": generation_seconds,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_validation.json",
        result,
    )
    write_json(
        VALIDATION_DIR / f"epoch_{epoch:02d}_predictions.json",
        {
            str(image_id): record
            for image_id, record in records.items()
        },
    )
    return result


def selection_key(result):
    return (
        float(result["metrics"]["CIDEr-D"]),
        float(result["metrics"]["BLEU-4"]),
        -float(result["teacher_forced_ce"]),
        -int(result["epoch"]),
    )


def append_step_log(row):
    with STEP_LOG.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )


def write_epoch_history(history):
    if not history:
        return
    rows = []
    for result in history:
        rows.append(
            {
                "epoch": result["epoch"],
                "train_combined_loss": result["train_combined_loss"],
                "train_human_ce": result["train_human_ce"],
                "train_teacher_sequence_ce": result[
                    "train_teacher_sequence_ce"
                ],
                "validation_ce": result["teacher_forced_ce"],
                "BLEU-1": result["metrics"]["BLEU-1"],
                "BLEU-4": result["metrics"]["BLEU-4"],
                "METEOR": result["metrics"]["METEOR"],
                "ROUGE-L": result["metrics"]["ROUGE-L"],
                "CIDEr-D": result["metrics"]["CIDEr-D"],
                "generation_seconds": result["generation_seconds"],
            }
        )
    with EPOCH_HISTORY_CSV.open(
        "w", encoding="utf-8", newline=""
    ) as handle:
        writer = csv.DictWriter(
            handle, fieldnames=list(rows[0])
        )
        writer.writeheader()
        writer.writerows(rows)


optimizer, scheduler, scaler = build_optimizer_state()
progress = {
    "epoch": 0,
    "cursor": 0,
    "global_optimizer_step": 0,
    "epoch_accumulators": {
        "human_loss_sum": 0.0,
        "teacher_loss_sum": 0.0,
        "human_tokens": 0,
        "teacher_tokens": 0,
        "overflow_retries": 0,
    },
}
history = []
best_result = None
no_improve_epochs = 0
resume_source = None

if LAST_CHECKPOINT.is_file():
    resume_source = LAST_CHECKPOINT
elif INPUT_RESUME_CHECKPOINT is not None:
    resume_source = INPUT_RESUME_CHECKPOINT
    previous_root = resume_source.parent.parent
    for previous_directory, current_directory in [
        (previous_root / "logs", LOG_DIR),
        (previous_root / "validation", VALIDATION_DIR),
    ]:
        if previous_directory.is_dir():
            for previous_file in previous_directory.glob("*"):
                if previous_file.is_file():
                    current_file = (
                        current_directory / previous_file.name
                    )
                    if not current_file.exists():
                        shutil.copy2(previous_file, current_file)
    previous_best = resume_source.parent / BEST_CHECKPOINT.name
    if previous_best.is_file() and not BEST_CHECKPOINT.exists():
        shutil.copy2(previous_best, BEST_CHECKPOINT)

if resume_source is not None:
    progress, history, best_result, no_improve_epochs = (
        load_checkpoint(
            resume_source, optimizer, scheduler, scaler
        )
    )
    print("Resumed from              :", resume_source)
    print("Resume progress          :", progress)
else:
    print("Starting from locked Student initialization")

# Establish a recoverable optimizer-step boundary before entering the loop.
# On manual interruption, restore this last known-good checkpoint instead of
# serializing a potentially half-computed microbatch.
save_checkpoint(
    LAST_CHECKPOINT,
    optimizer, scheduler, scaler,
    dict(progress), history, best_result,
    no_improve_epochs,
)

TRAINING_COMPLETED = False
EARLY_STOPPED = False
INTERRUPTED_SAFELY = False
training_started_at = time.perf_counter()
SESSION_BUDGET_SECONDS = 10.5 * 3600
SESSION_BUDGET_REACHED = False


class SessionBudgetReached(Exception):
    pass


try:
    while int(progress["epoch"]) < MAX_EPOCHS:
        epoch_index = int(progress["epoch"])
        epoch_number = epoch_index + 1
        cursor = int(progress["cursor"])
        loader = build_train_loader(epoch_index, cursor=cursor)
        iterator = iter(loader)
        accumulator = progress.get(
            "epoch_accumulators",
            {
                "human_loss_sum": 0.0,
                "teacher_loss_sum": 0.0,
                "human_tokens": 0,
                "teacher_tokens": 0,
                "overflow_retries": 0,
            },
        )
        epoch_human_loss_sum = float(
            accumulator["human_loss_sum"]
        )
        epoch_teacher_loss_sum = float(
            accumulator["teacher_loss_sum"]
        )
        epoch_human_tokens = int(accumulator["human_tokens"])
        epoch_teacher_tokens = int(
            accumulator["teacher_tokens"]
        )
        epoch_images = cursor
        epoch_overflows = int(
            accumulator["overflow_retries"]
        )

        while True:
            microbatches = []
            for _ in range(ACCUMULATION):
                try:
                    microbatches.append(next(iterator))
                except StopIteration:
                    break
            if not microbatches:
                break
            result = optimizer_step(
                microbatches, optimizer, scheduler, scaler,
                retry_limit=10,
            )
            images = sum(
                len(batch["image_ids"]) for batch in microbatches
            )
            epoch_images += images
            progress["cursor"] = epoch_images
            progress["global_optimizer_step"] = (
                int(progress["global_optimizer_step"]) + 1
            )
            epoch_human_loss_sum += (
                result["human_ce"] * result["human_tokens"]
            )
            epoch_teacher_loss_sum += (
                result["teacher_sequence_ce"]
                * result["teacher_tokens"]
            )
            epoch_human_tokens += result["human_tokens"]
            epoch_teacher_tokens += result["teacher_tokens"]
            epoch_overflows += result["overflow_retries"]
            progress["epoch_accumulators"] = {
                "human_loss_sum": epoch_human_loss_sum,
                "teacher_loss_sum": epoch_teacher_loss_sum,
                "human_tokens": epoch_human_tokens,
                "teacher_tokens": epoch_teacher_tokens,
                "overflow_retries": epoch_overflows,
            }
            step_row = {
                "created_at_utc": now_utc(),
                "epoch": epoch_number,
                "cursor": epoch_images,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
                **result,
                "lr_backbone": optimizer.param_groups[0]["lr"],
                "lr_adapter": optimizer.param_groups[1]["lr"],
                "lr_decoder": optimizer.param_groups[2]["lr"],
            }
            append_step_log(step_row)
            if (
                int(progress["global_optimizer_step"]) % 50 == 0
            ):
                print(
                    f"epoch={epoch_number:02d} | "
                    f"step={progress['global_optimizer_step']:05d} | "
                    f"cursor={epoch_images:06d} | "
                    f"loss={result['combined_loss']:.6f}"
                )
            if (
                int(progress["global_optimizer_step"]) % 250 == 0
            ):
                save_checkpoint(
                    LAST_CHECKPOINT,
                    optimizer, scheduler, scaler,
                    dict(progress), history, best_result,
                    no_improve_epochs,
                )
            if (
                time.perf_counter() - training_started_at
                >= SESSION_BUDGET_SECONDS
            ):
                save_checkpoint(
                    LAST_CHECKPOINT,
                    optimizer, scheduler, scaler,
                    dict(progress), history, best_result,
                    no_improve_epochs,
                )
                SESSION_BUDGET_REACHED = True
                raise SessionBudgetReached
        assert epoch_images == 113287
        assert epoch_human_tokens > 0
        assert epoch_teacher_tokens > 0

        validation = evaluate_validation(epoch_number)
        validation.update(
            {
                "train_human_ce": (
                    epoch_human_loss_sum / epoch_human_tokens
                ),
                "train_teacher_sequence_ce": (
                    epoch_teacher_loss_sum / epoch_teacher_tokens
                ),
                "train_combined_loss": (
                    (1 - ALPHA)
                    * epoch_human_loss_sum / epoch_human_tokens
                    + ALPHA
                    * epoch_teacher_loss_sum / epoch_teacher_tokens
                ),
                "train_human_tokens": epoch_human_tokens,
                "train_teacher_tokens": epoch_teacher_tokens,
                "train_images": epoch_images,
                "overflow_retries": epoch_overflows,
                "global_optimizer_step": progress[
                    "global_optimizer_step"
                ],
            }
        )
        history.append(validation)

        previous_best_cider = (
            float(best_result["metrics"]["CIDEr-D"])
            if best_result is not None else float("-inf")
        )
        if (
            best_result is None
            or selection_key(validation) > selection_key(best_result)
        ):
            best_result = copy.deepcopy(validation)
            save_checkpoint(
                BEST_CHECKPOINT,
                optimizer, scheduler, scaler,
                {
                    "epoch": epoch_index + 1,
                    "cursor": 0,
                    "global_optimizer_step": progress[
                        "global_optimizer_step"
                    ],
                },
                history, best_result, no_improve_epochs,
            )

        if (
            float(validation["metrics"]["CIDEr-D"])
            >= previous_best_cider + 0.001
        ):
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1

        progress["epoch"] = epoch_index + 1
        progress["cursor"] = 0
        progress["epoch_accumulators"] = {
            "human_loss_sum": 0.0,
            "teacher_loss_sum": 0.0,
            "human_tokens": 0,
            "teacher_tokens": 0,
            "overflow_retries": 0,
        }
        save_checkpoint(
            LAST_CHECKPOINT,
            optimizer, scheduler, scaler,
            dict(progress), history, best_result,
            no_improve_epochs,
        )
        write_epoch_history(history)
        print(
            f"epoch={epoch_number:02d} COMPLETE | "
            f"CIDEr-D={validation['metrics']['CIDEr-D']:.6f} | "
            f"best={best_result['metrics']['CIDEr-D']:.6f} | "
            f"CE={validation['teacher_forced_ce']:.6f}"
        )
        if (
            epoch_number >= 5
            and no_improve_epochs >= 4
        ):
            EARLY_STOPPED = True
            break

    TRAINING_COMPLETED = True
except (KeyboardInterrupt, SessionBudgetReached):
    (
        progress,
        history,
        best_result,
        no_improve_epochs,
    ) = load_checkpoint(
        LAST_CHECKPOINT, optimizer, scheduler, scaler
    )
    INTERRUPTED_SAFELY = True
    print(
        "Session budget reached."
        if SESSION_BUDGET_REACHED
        else "Manual interruption captured."
    )
    print("Restored last verified optimizer-step boundary:", progress)

training_seconds_this_session = (
    time.perf_counter() - training_started_at
)
write_json(
    RUN_STATE_PATH,
    {
        "schema_version": "phase4l_seed2026_run_state_v1",
        "created_at_utc": now_utc(),
        "status": (
            "complete" if TRAINING_COMPLETED
            else "interrupted_resume_ready"
        ),
        "progress": progress,
        "history_epochs": len(history),
        "best_result": best_result,
        "early_stopped": EARLY_STOPPED,
        "interrupted_safely": INTERRUPTED_SAFELY,
        "session_budget_reached": SESSION_BUDGET_REACHED,
        "training_seconds_this_session": training_seconds_this_session,
        "last_checkpoint": str(LAST_CHECKPOINT),
        "last_checkpoint_sha256": (
            sha256_file(LAST_CHECKPOINT)
            if LAST_CHECKPOINT.is_file() else None
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 5 STATUS")
print("=" * 72)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Global optimizer steps    :", progress["global_optimizer_step"])
print("Current cursor            :", progress["cursor"])
print("Best epoch                :", (
    best_result["epoch"] if best_result else None
))
print("Best CIDEr-D              :", (
    best_result["metrics"]["CIDEr-D"] if best_result else None
))
print("Early stopped             :", EARLY_STOPPED)
print("Interrupted safely        :", INTERRUPTED_SAFELY)
print("Session budget reached    :", SESSION_BUDGET_REACHED)
print("Last checkpoint           :", LAST_CHECKPOINT)
print("Test Set status           : SEALED")
print("Next                      : package complete or resume state")


Starting from locked Student initialization


/tmp/ipykernel_23/2563310880.py:278: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  human_sum_total += float(human_sum)


epoch=01 | step=00050 | cursor=003200 | loss=10.156590
epoch=01 | step=00100 | cursor=006400 | loss=7.902101
epoch=01 | step=00150 | cursor=009600 | loss=6.558162
epoch=01 | step=00200 | cursor=012800 | loss=5.729927
epoch=01 | step=00250 | cursor=016000 | loss=5.202524
epoch=01 | step=00300 | cursor=019200 | loss=4.587117
epoch=01 | step=00350 | cursor=022400 | loss=4.226385
epoch=01 | step=00400 | cursor=025600 | loss=4.193746
epoch=01 | step=00450 | cursor=028800 | loss=3.921223
epoch=01 | step=00500 | cursor=032000 | loss=3.655274
epoch=01 | step=00550 | cursor=035200 | loss=3.665615
epoch=01 | step=00600 | cursor=038400 | loss=3.614727
epoch=01 | step=00650 | cursor=041600 | loss=3.666732
epoch=01 | step=00700 | cursor=044800 | loss=3.360579
epoch=01 | step=00750 | cursor=048000 | loss=3.217301
epoch=01 | step=00800 | cursor=051200 | loss=2.955374
epoch=01 | step=00850 | cursor=054400 | loss=3.059485
epoch=01 | step=00900 | cursor=057600 | loss=2.986227
epoch=01 | step=00950 | cur

PTBTokenizer tokenized 307821 tokens at 798114.45 tokens per second.
PTBTokenizer tokenized 52808 tokens at 266198.06 tokens per second.


{'testlen': 47809, 'reflen': 47448, 'guess': [47809, 42809, 37809, 32809], 'correct': [30987, 14405, 6329, 2965]}
ratio: 1.0076083291181714
epoch=01 COMPLETE | CIDEr-D=0.716792 | best=0.716792 | CE=3.146305
epoch=02 | step=01800 | cursor=001856 | loss=2.560854
epoch=02 | step=01850 | cursor=005056 | loss=2.346936
epoch=02 | step=01900 | cursor=008256 | loss=2.323179
epoch=02 | step=01950 | cursor=011456 | loss=2.263182
epoch=02 | step=02000 | cursor=014656 | loss=2.144497
epoch=02 | step=02050 | cursor=017856 | loss=2.220733
epoch=02 | step=02100 | cursor=021056 | loss=2.233626
epoch=02 | step=02150 | cursor=024256 | loss=2.139274
epoch=02 | step=02200 | cursor=027456 | loss=2.296703
epoch=02 | step=02250 | cursor=030656 | loss=2.242299
epoch=02 | step=02300 | cursor=033856 | loss=2.106404
epoch=02 | step=02350 | cursor=037056 | loss=2.339705
epoch=02 | step=02400 | cursor=040256 | loss=2.186634
epoch=02 | step=02450 | cursor=043456 | loss=2.014497
epoch=02 | step=02500 | cursor=046656

PTBTokenizer tokenized 307821 tokens at 717152.45 tokens per second.
PTBTokenizer tokenized 53989 tokens at 338495.58 tokens per second.


... (33 outputs left out)

PTBTokenizer tokenized 307821 tokens at 753118.11 tokens per second.
PTBTokenizer tokenized 54761 tokens at 276961.54 tokens per second.


{'testlen': 49762, 'reflen': 48951, 'guess': [49762, 44762, 39762, 34762], 'correct': [35764, 19250, 9845, 5109]}
ratio: 1.0165675879961387
epoch=19 COMPLETE | CIDEr-D=1.023532 | best=1.024771 | CE=2.308109
epoch=20 | step=33650 | cursor=000064 | loss=1.253692
epoch=20 | step=33700 | cursor=003264 | loss=1.316505
epoch=20 | step=33750 | cursor=006464 | loss=1.291432
epoch=20 | step=33800 | cursor=009664 | loss=1.452169
epoch=20 | step=33850 | cursor=012864 | loss=1.360701
epoch=20 | step=33900 | cursor=016064 | loss=1.340217
epoch=20 | step=33950 | cursor=019264 | loss=1.393422
epoch=20 | step=34000 | cursor=022464 | loss=1.336736
epoch=20 | step=34050 | cursor=025664 | loss=1.509730
epoch=20 | step=34100 | cursor=028864 | loss=1.298943
epoch=20 | step=34150 | cursor=032064 | loss=1.440915
epoch=20 | step=34200 | cursor=035264 | loss=1.360654
epoch=20 | step=34250 | cursor=038464 | loss=1.408131
epoch=20 | step=34300 | cursor=041664 | loss=1.454634
epoch=20 | step=34350 | cursor=044864

PTBTokenizer tokenized 307821 tokens at 681932.89 tokens per second.
PTBTokenizer tokenized 54801 tokens at 281181.94 tokens per second.


{'testlen': 49802, 'reflen': 48976, 'guess': [49802, 44802, 39802, 34802], 'correct': [35753, 19272, 9851, 5114]}
ratio: 1.0168654034628999
epoch=20 COMPLETE | CIDEr-D=1.022344 | best=1.024771 | CE=2.308817
PHASE 04L SEED 2026 CELL 5 STATUS
Status                    : COMPLETE
Completed epochs          : 20
Global optimizer steps    : 35420
Current cursor            : 0
Best epoch                : 18
Best CIDEr-D              : 1.0247712956839135
Early stopped             : False
Interrupted safely        : False
Session budget reached    : False
Last checkpoint           : /kaggle/working//phase4_teacher_b_s2_sequence_kd_seed2026/checkpoints/teacher_b_s2_sequence_kd_seed2026_last.pt
Test Set status           : SEALED
Next                      : package complete or resume state


## Cell 6 — Package seed 2026 complete hoặc resumable state

Khi cả seed 123 và 2026 hoàn tất, chuyển sang Phase 04M aggregate.


In [6]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4l_seed2026_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4l_seed2026_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4l_seed2026_SHA256SUMS.txt"

if TRAINING_COMPLETED:
    assert best_result is not None
    assert BEST_CHECKPOINT.is_file()
    assert LAST_CHECKPOINT.is_file()
    kd_cider = float(best_result["metrics"]["CIDEr-D"])
    kd_bleu4 = float(best_result["metrics"]["BLEU-4"])
    kd_ce = float(best_result["teacher_forced_ce"])
    comparison = {
        "baseline_seed2026": {
            "CIDEr-D": BASELINE_CIDER,
            "BLEU-4": BASELINE_BLEU4,
            "validation_ce": BASELINE_CE,
        },
        "kd_seed2026": {
            "CIDEr-D": kd_cider,
            "BLEU-4": kd_bleu4,
            "validation_ce": kd_ce,
            "best_epoch": int(best_result["epoch"]),
        },
        "delta": {
            "CIDEr-D": kd_cider - BASELINE_CIDER,
            "BLEU-4": kd_bleu4 - BASELINE_BLEU4,
            "validation_ce": kd_ce - BASELINE_CE,
        },
        "kd_improves_cider_by_0_001": (
            kd_cider >= BASELINE_CIDER + 0.001
        ),
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    COMPARISON_PATH = RESULT_DIR / (
        "teacher_b_s2_sequence_kd_seed2026_vs_baseline.json"
    )
    write_json(COMPARISON_PATH, comparison)
    NEXT_AUTHORIZATION = None
    FINAL_MARKER = OUTPUT_ROOT / (
        "PHASE4_TEACHER_B_S2_SEQUENCE_KD_SEED2026_FULL_RUN_COMPLETE.json"
    )
else:
    assert LAST_CHECKPOINT.is_file()
    NEXT_AUTHORIZATION = None
    comparison = None
    FINAL_MARKER = OUTPUT_ROOT / (
        "PHASE4_TEACHER_B_S2_SEQUENCE_KD_SEED2026_RESUME_READY.json"
    )

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4l_seed2026_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

marker_payload = {
    "marker_version": (
        "phase4_teacher_b_s2_sequence_kd_seed2026_"
        + ("full_run_complete_v1" if TRAINING_COMPLETED
           else "resume_ready_v1")
    ),
    "created_at_utc": now_utc(),
    "status": "complete" if TRAINING_COMPLETED else "resume_ready",
    "experiment_id": "P4B_B_S2_SEQKD",
    "seed": SEED,
    "progress": progress,
    "completed_epochs": len(history),
    "best_result": best_result,
    "last_checkpoint": {
        "relative_path": LAST_CHECKPOINT.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(LAST_CHECKPOINT),
    },
    "best_checkpoint": (
        {
            "relative_path": BEST_CHECKPOINT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(BEST_CHECKPOINT),
        }
        if BEST_CHECKPOINT.is_file() else None
    ),
    "comparison": comparison,
    "ready_for_phase04m_three_seed_aggregate": bool(TRAINING_COMPLETED),
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
        },
        "json": {
            "relative_path": MANIFEST_JSON.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_JSON),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS),
        },
    },
    "manifest_rows": len(manifest_rows),
    "manifest_verification": "all_passed",
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(FINAL_MARKER, marker_payload)

print("=" * 72)
print("PHASE 04L SEED 2026 CELL 6 STATUS")
print("=" * 72)
print(
    "Status                    :",
    "COMPLETE" if TRAINING_COMPLETED else "RESUME READY",
)
print("Completed epochs          :", len(history))
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Last checkpoint           : VERIFIED")
print("Best checkpoint           :", BEST_CHECKPOINT.is_file())
print(
    "Phase 04M aggregate      :",
    "READY" if TRAINING_COMPLETED else "NOT YET",
)
print("Test Set status           : SEALED")
print("Marker                    :", FINAL_MARKER)
print(
    "Next                      :",
    (
        "After seed 123 and 2026 complete: Phase 04M aggregate"
        if TRAINING_COMPLETED
        else "Save Version, attach output, rerun seed 2026"
    ),
)


PHASE 04L SEED 2026 CELL 6 STATUS
Status                    : COMPLETE
Completed epochs          : 20
Manifest rows             : 52
Manifest verification     : ALL PASSED
Last checkpoint           : VERIFIED
Best checkpoint           : True
Phase 04M aggregate      : READY
Test Set status           : SEALED
Marker                    : /kaggle/working//phase4_teacher_b_s2_sequence_kd_seed2026/PHASE4_TEACHER_B_S2_SEQUENCE_KD_SEED2026_FULL_RUN_COMPLETE.json
Next                      : After seed 123 and 2026 complete: Phase 04M aggregate
